# 雾密度网络：codex-v1 最简实验流程

更新于 2026-10-21，适用于 `codex-v1` 分支和先验版本 `spatial_minmax_valid_boundary_v3`。

`准备代码 → 小张量检查 → 准备数据 → 固定实验配置和划分 → 训练 → 验证集看图 → 可选测试集评估`

配置、划分、权重、日志和结果保存到 Google Drive；数据解压到临时的 `/content`。运行时被回收后，重新准备环境和数据，再使用原实验目录恢复。

网络学习的是三先验生成的**相对评分伪标签**。与先验之间的误差衡量规则拟合，不能独立证明真实物理雾密度或下游去雾、检测效果。此 notebook 不构建阶段 2 Benchmark。

## 1. 环境初始化

在 Colab 中选择 GPU 运行时，然后运行一次。本节始终挂载 Drive，并明确获取 `codex-v1`，打印代码提交号和先验版本。

已有的干净仓库会切换到该分支并仅以快进方式更新。若有本地修改，先保留修改；已导入项目后需要更新代码时，请重启运行时，避免内存中继续使用旧模块。同一运行时完成初始化后，从后续步骤继续。

请先把本次修改推送到远端 `codex-v1`，再在 Colab 运行；只上传 notebook 不会更新训练源码。

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
from datetime import datetime, timezone
from google.colab import drive

PROJECT_ROOT = Path('/content/CrossModal-Defog-OVOD')
DRIVE_ROOT = Path('/content/drive')
REPO_URL = 'https://github.com/codeGease-1/CrossModal-Defog-OVOD.git'
BRANCH = 'codex-v1'
EXPECTED_PRIOR_VERSION = 'spatial_minmax_valid_boundary_v3'

# 实时显示子进程日志；失败立即停止，避免把旧文件当作本次运行成功。
def run_command(args, cwd=None):
    command = [str(arg) for arg in args]
    with subprocess.Popen(
        command, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding='utf-8', bufsize=1,
        env={**os.environ, 'PYTHONUNBUFFERED': '1', 'PYTHONIOENCODING': 'utf-8'},
    ) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        returncode = process.wait()
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)

# 挂载与项目目录是否存在无关。
drive.mount(str(DRIVE_ROOT))
if any(name == 'src' or name.startswith('src.') for name in sys.modules):
    raise RuntimeError('当前运行时已导入项目。继续运行后续步骤；如需更新代码，请先重启运行时。')

if not PROJECT_ROOT.exists():
    run_command(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, PROJECT_ROOT])
else:
    if not (PROJECT_ROOT / '.git').is_dir():
        raise RuntimeError(f'{PROJECT_ROOT} 不是 Git 仓库，请使用新的 Colab 运行时。')
    changes = subprocess.check_output(
        ['git', '-C', str(PROJECT_ROOT), 'status', '--porcelain'], text=True,
    ).strip()
    if changes:
        raise RuntimeError('仓库有本地修改，请先保留修改，再更新分支。\n' + changes)
    run_command(['git', 'fetch', 'origin', f'refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}'], cwd=PROJECT_ROOT)
    run_command(['git', 'switch', BRANCH], cwd=PROJECT_ROOT)
    run_command(['git', 'merge', '--ff-only', f'origin/{BRANCH}'], cwd=PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
CODE_BRANCH = subprocess.check_output(['git', 'branch', '--show-current'], text=True).strip()
CODE_COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
if CODE_BRANCH != BRANCH:
    raise RuntimeError(f'分支错误：{CODE_BRANCH}，预期 {BRANCH}')
run_command([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
import yaml
from src.models.haze_density.physical_prior import PRIOR_VERSION
if PRIOR_VERSION != EXPECTED_PRIOR_VERSION:
    raise RuntimeError(f'先验版本不符：{PRIOR_VERSION}，请确认远端分支包含阶段 1 修复。')
print(f'Branch: {CODE_BRANCH}\nCommit: {CODE_COMMIT}\nPrior: {PRIOR_VERSION}')
print(f'PyTorch: {torch.__version__}; CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. 训练前的小张量检查

每次更换代码或运行时后执行。这里只运行回归与前向、反向检查，不进行正式训练，也不需要 RSHaze+。检查失败时先解决报错；`skipped` 不代表通过。

In [ ]:
checks = [
    ['scripts/utils/static_check.py'],
    ['-m', 'unittest', 'discover', '-s', 'scripts/test', '-p', 'test_review_regressions.py', '-v'],
    ['-m', 'unittest', 'discover', '-s', 'scripts/test', '-p', 'test_stage1_bugfixes.py', '-v'],
    ['scripts/test/test_physical_prior.py'],
    ['scripts/utils/smoke_test.py'],
]
for check in checks:
    run_command([sys.executable, *check], cwd=PROJECT_ROOT)

## 3. 数据准备

先把 `RSHaze+.zip` 放到 Drive 的 `MyDrive/dataset/`。已有解压目录时跳过解压，并检查三个子集的官方 train/test 目录。

最终结构为 `datasets/RSHaze+/RSHaze_G/train/synhazypng/` 等，同时有对应的 `cleanpng/`；另外两个子集为 `RSHaze_L`、`RSHaze_S`。当前密度训练使用有雾图，清晰图不参与 MSE 目标。若检查失败，先核对压缩包层级和数据完整性。

In [ ]:
import shutil

DATASETS_DIR = PROJECT_ROOT / 'datasets'
DATASET_ROOT = DATASETS_DIR / 'RSHaze+'
DRIVE_ZIP = DRIVE_ROOT / 'MyDrive/dataset/RSHaze+.zip'
SUBSETS = ('RSHaze_G', 'RSHaze_L', 'RSHaze_S')

if not DATASET_ROOT.exists():
    if not DRIVE_ZIP.is_file():
        raise FileNotFoundError(f'找不到 {DRIVE_ZIP}，请准备压缩包或手动放置完整数据集。')
    DATASETS_DIR.mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(DRIVE_ZIP, DATASETS_DIR)
    if not DATASET_ROOT.exists():
        nested = DATASETS_DIR / 'RSHazePlus' / 'RSHaze+'
        if not nested.is_dir():
            raise FileNotFoundError('解压后未找到 RSHaze+，请核对压缩包目录层级。')
        shutil.copytree(nested, DATASET_ROOT)
else:
    print(f'沿用已有数据目录：{DATASET_ROOT}')

for subset in SUBSETS:
    for official_split in ('train', 'test'):
        for folder in ('synhazypng', 'cleanpng'):
            directory = DATASET_ROOT / subset / official_split / folder
            if not directory.is_dir() or not any(directory.glob('*.png')):
                raise FileNotFoundError(f'目录缺失或没有 PNG：{directory}')
    print(f'{subset}: 官方 train/test 目录检查通过')

## 4. 固定实验配置和数据划分

首次实验只需修改 `RUN_NAME` 和下面的训练设置。配置从当前 YAML 读取，路径指向 Drive；同名实验再次运行时沿用保存的配置与划分。要修改这些设置并从头训练，请换一个 `RUN_NAME`。

`RUN_NAME` 应是单个目录名，例如 `spatial_v3_run01`。数据不变时不重新划分；清单会检查路径重复、覆盖情况和数据是否匹配，这不等于排除了内容重复或同场景泄漏。

In [ ]:
from src.data import HazeDensityDataset
from scripts.data.generate_rshazeplus_split import generate_split
from scripts.data.verify_rshazeplus_split import verify_split_json

RUN_NAME = 'spatial_v3_run01'
IMAGE_SIZE = 256
BATCH_SIZE = 4
LEARNING_RATE = 1e-4

if not RUN_NAME or RUN_NAME in ('.', '..') or '/' in RUN_NAME or chr(92) in RUN_NAME:
    raise ValueError('RUN_NAME 请使用单个目录名。')
RUN_DIR = DRIVE_ROOT / 'MyDrive/CrossModal-Defog-OVOD' / RUN_NAME
CONFIG_PATH = RUN_DIR / 'train_config.yaml'
RUN_DIR.mkdir(parents=True, exist_ok=True)

requested = yaml.safe_load((PROJECT_ROOT / 'configs/haze_density.yaml').read_text(encoding='utf-8'))
requested.update(
    dataset_root=str(DATASET_ROOT),
    split_file=str(RUN_DIR / 'rshazeplus_split.json'),
    checkpoint_dir=str(RUN_DIR / 'checkpoints'),
    result_dir=str(RUN_DIR / 'results'),
    image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, lr=LEARNING_RATE,
)
if CONFIG_PATH.exists():
    TRAIN_CONFIG = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
    changed = [key for key in requested if TRAIN_CONFIG.get(key) != requested[key]]
    if changed:
        raise ValueError(f'已有配置与当前设置不同：{changed}。请还原设置以恢复，或换 RUN_NAME 开新实验。')
    print(f'沿用配置：{CONFIG_PATH}')
else:
    TRAIN_CONFIG = requested
    CONFIG_PATH.write_text(yaml.safe_dump(TRAIN_CONFIG, allow_unicode=True, sort_keys=False), encoding='utf-8')

# 与 YAML 分开保存，避免给训练脚本传入不支持的配置字段。
code_info = {'branch': CODE_BRANCH, 'commit': CODE_COMMIT, 'prior_version': PRIOR_VERSION}
(RUN_DIR / f'code_{CODE_COMMIT[:12]}.json').write_text(
    json.dumps(code_info, ensure_ascii=False, indent=2) + '\n', encoding='utf-8',
)
SPLIT_PATH = Path(TRAIN_CONFIG['split_file'])
CHECKPOINT_DIR = Path(TRAIN_CONFIG['checkpoint_dir'])
if not SPLIT_PATH.exists():
    if any((CHECKPOINT_DIR / name).exists() for name in ('best.pth', 'latest.pth')):
        raise FileNotFoundError('已有权重但缺少原划分，请找回该实验的 manifest，或开新实验。')
    _, success = generate_split(
        root=TRAIN_CONFIG['dataset_root'], val_ratio=0.1, seed=42,
        output_file=str(SPLIT_PATH),
    )
    if not success:
        raise RuntimeError('生成数据划分失败。')
if not verify_split_json(str(SPLIT_PATH)):
    raise RuntimeError('已有数据划分检查失败，请核对清单和原始数据。')

manifest = json.loads(SPLIT_PATH.read_text(encoding='utf-8'))
for split in ('train', 'val', 'test'):
    dataset = HazeDensityDataset(
        root=TRAIN_CONFIG['dataset_root'], split=split,
        image_size=TRAIN_CONFIG['image_size'], split_file=str(SPLIT_PATH),
    )
    actual = {(item['subset'], item['filename']) for item in dataset.image_list}
    recorded = {(item['subset'], item['filename']) for item in manifest[split]}
    if actual != recorded:
        raise ValueError(f'{split} 清单与实际数据不一致，请检查数据版本。')
    if split == 'train' and len(dataset) < TRAIN_CONFIG['batch_size']:
        raise ValueError('训练样本少于 batch_size，会导致训练 DataLoader 为空。')
    print(f'{split}: {len(dataset)} 张')
print(f'实验目录：{RUN_DIR}\n配置：{CONFIG_PATH}')

## 5. 训练与断点续训

第一次使用 `EPOCHS = 5`、`RESUME = False` 做短训练。正常完成后，将 `EPOCHS` 改为目标总轮数（例如 50），`RESUME = True`，从本实验的 `latest.pth` 继续。

`EPOCHS = 50` 表示累计训练到第 50 轮。恢复轮数必须大于已保存轮数。已有实验不能再次从头覆盖；新实验请更改第 4 节的 `RUN_NAME` 并重新执行第 4 节。

旧的无版本标记或 v2 权重不能续训成同一个 v3 实验。`best.pth` 用于查看验证集最优结果，`latest.pth` 用于恢复。日志逐轮保存在 Drive 的 `results/train_log.csv`。完整训练建议在 GPU 运行时执行。

In [ ]:
EPOCHS = 5
RESUME = False

command = [
    sys.executable, '-u', 'scripts/train/train_haze_density.py',
    '--config', str(CONFIG_PATH), '--epochs', str(EPOCHS),
]
if RESUME:
    resume_path = CHECKPOINT_DIR / 'latest.pth'
    if not resume_path.is_file():
        raise FileNotFoundError(f'找不到续训权重：{resume_path}')
    saved = torch.load(resume_path, map_location='cpu')
    if saved.get('config', {}).get('prior_version') != PRIOR_VERSION:
        raise ValueError('权重的先验版本不匹配，请为 v3 新开实验。')
    if EPOCHS <= saved['epoch']:
        raise ValueError(f'已训练 {saved["epoch"]} 轮，请将 EPOCHS 设为更大的目标总轮数。')
    command += ['--resume', str(resume_path)]
else:
    occupied = any((CHECKPOINT_DIR / name).exists() for name in ('best.pth', 'latest.pth'))
    if occupied or (Path(TRAIN_CONFIG['result_dir']) / 'train_log.csv').exists():
        raise FileExistsError('该实验已有权重或日志；请设 RESUME=True，或使用新的 RUN_NAME。')
run_command(command, cwd=PROJECT_ROOT)

BEST_CHECKPOINT = CHECKPOINT_DIR / 'best.pth'
if not BEST_CHECKPOINT.is_file():
    raise FileNotFoundError(f'训练结束但没有找到 {BEST_CHECKPOINT}')
print(f'本次训练命令成功结束，最优权重：{BEST_CHECKPOINT}')

## 6. 加载 best.pth，在验证集上看图

本节可以单独重复运行。运行时重建后，先完成第 1—4 节以恢复路径，再直接运行本节，无需重新训练。

显示 `含雾图 | 先验伪标签 | 网络预测 | 绝对误差`，每子集最多 2 张。图像尺寸、模型和先验参数都从 checkpoint 读取；色标固定 0—1，每次保存到 Drive 的新时间目录。

这些图是按数据遍历顺序取样，用来检查训练结果；不能代替代表性抽样或独立正确性评测。日常调参使用验证集，测试集留到第 7 节。

In [ ]:
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt
from src.data import build_rshazeplus_dataloader
from src.models.haze_density import HazeDensityNet
from src.models.haze_density.physical_prior import PhysicalPriorModule

BEST_CHECKPOINT = CHECKPOINT_DIR / 'best.pth'
if not BEST_CHECKPOINT.is_file():
    raise FileNotFoundError(f'找不到 {BEST_CHECKPOINT}，请先完成本实验的训练。')
checkpoint = torch.load(BEST_CHECKPOINT, map_location='cpu')
checkpoint_config = checkpoint['config']
if checkpoint_config.get('prior_version') != PRIOR_VERSION:
    raise ValueError('此 notebook 仅展示当前 v3 先验实验，请保留旧权重作历史资料。')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = HazeDensityNet(
    base_channels=checkpoint_config['base_channels'],
    use_sigmoid=checkpoint_config['use_sigmoid'],
).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
physical_prior = PhysicalPriorModule(
    window_size=checkpoint_config['window_size'],
    guided_radius=checkpoint_config['guided_radius'],
    guided_eps=checkpoint_config['guided_eps'],
).to(device).eval()
preview_loader = build_rshazeplus_dataloader(
    root=checkpoint_config['dataset_root'], split='val',
    image_size=checkpoint_config['image_size'],
    batch_size=TRAIN_CONFIG['batch_size'], num_workers=0,
    pin_memory=(device.type == 'cuda'), split_file=checkpoint_config['split_file'],
)

stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
output_dir = RUN_DIR / 'validation_previews' / stamp
output_dir.mkdir(parents=True, exist_ok=False)
max_per_subset = 2
saved = defaultdict(int)
with torch.inference_mode():
    for batch in preview_loader:
        images = batch['image'].to(device)
        predictions = model(images)
        priors = physical_prior(images)
        if not torch.isfinite(predictions).all() or not torch.isfinite(priors).all():
            raise ValueError('预测或先验出现 NaN/Inf，请先排查，停止保存本次结果。')
        for i, subset in enumerate(batch['subset']):
            if saved[subset] >= max_per_subset:
                continue
            image = images[i].cpu().permute(1, 2, 0).numpy()
            prior = priors[i, 0].cpu().numpy()
            prediction = predictions[i, 0].cpu().numpy()
            error = np.abs(prior - prediction)
            fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
            axes[0].imshow(image)
            axes[0].set_title('Hazy image')
            axes[1].imshow(prior, cmap='magma', vmin=0, vmax=1)
            axes[1].set_title('Prior pseudo-label')
            axes[2].imshow(prediction, cmap='magma', vmin=0, vmax=1)
            axes[2].set_title('Network prediction')
            axes[3].imshow(error, cmap='viridis', vmin=0, vmax=1)
            axes[3].set_title(f'Absolute error (MAE={error.mean():.4f})')
            for axis in axes:
                axis.axis('off')
            fig.suptitle(f'Validation: {subset} / {batch["filename"][i]}')
            fig.tight_layout()
            output_path = output_dir / f'{subset}_{Path(batch["filename"][i]).stem}_density_result.png'
            fig.savefig(output_path, dpi=160, bbox_inches='tight')
            plt.show()
            plt.close(fig)
            saved[subset] += 1
        if all(saved[subset] >= max_per_subset for subset in SUBSETS):
            break

preview_info = {
    'checkpoint': str(BEST_CHECKPOINT), 'epoch': checkpoint['epoch'],
    'prior_version': PRIOR_VERSION, 'split': 'val',
    'image_size': checkpoint_config['image_size'],
    'prior_parameters': {key: checkpoint_config[key] for key in ('window_size', 'guided_radius', 'guided_eps')},
    'saved_per_subset': dict(saved),
}
(output_dir / 'preview_info.json').write_text(
    json.dumps(preview_info, ensure_ascii=False, indent=2) + '\n', encoding='utf-8',
)
print(f'Checkpoint: {BEST_CHECKPOINT}\n保存目录：{output_dir}\n每子集数量：{dict(saved)}')

## 7. 可选：方案确定后评估测试集

若只想检查训练是否正常，到第 6 节即可。最终方案确定后，再运行本节；不要反复用测试集挑参数。

评估使用 checkpoint 中的先验参数，并显式传递该实验的数据路径、划分和图像尺寸；每次使用新的空输出目录。MAE、RMSE、Pearson 衡量的是相对伪标签的拟合，不能单独作为物理雾密度准确性的证据。

In [ ]:
BEST_CHECKPOINT = CHECKPOINT_DIR / 'best.pth'
checkpoint = torch.load(BEST_CHECKPOINT, map_location='cpu')
evaluation_config = checkpoint['config']
if evaluation_config.get('prior_version') != PRIOR_VERSION:
    raise ValueError('权重先验版本不匹配，本流程不混用旧实验。')
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
evaluation_dir = RUN_DIR / 'test_evaluations' / stamp
run_command([
    sys.executable, '-u', 'scripts/evaluate/evaluate_haze_density.py',
    '--checkpoint', str(BEST_CHECKPOINT),
    '--dataset_root', evaluation_config['dataset_root'],
    '--split_file', evaluation_config['split_file'],
    '--image_size', str(evaluation_config['image_size']),
    '--batch_size', str(TRAIN_CONFIG['batch_size']),
    '--num_samples_per_subset', '16',
    '--output_dir', str(evaluation_dir),
], cwd=PROJECT_ROOT)
print(f'测试集报告与图像：{evaluation_dir}')

## 如何重复运行

- **新实验**：按顺序运行第 1—6 节；每次从头训练使用新的 `RUN_NAME`。
- **同一运行时续训**：第 5 节设 `RESUME=True`，增大 `EPOCHS` 后运行，再运行第 6 节。
- **新运行时恢复旧实验**：第 1—4 节使用原 `RUN_NAME` 和原设置，然后按上条续训。Drive 中的配置、划分和 checkpoint 会保留。
- **只看已有权重**：同一运行时直接运行第 6 节；新运行时先运行第 1—4 节，再运行第 6 节。
- **最终测试**：方案确定后运行第 7 节，并保留本次报告和输出目录。

更多说明见 [操作指南](guides/haze_density_user_guide.md)、[阶段 1 修复报告](stage_1_bugfix_report.md)和 [Notebook 迁移说明](notebook_update_guide.md)。